# BERT Approach

This notebook presents the bert based approach in solving the three way classification problem of clarity labeling

In [1]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from torch.optim import AdamW
import tqdm

In [2]:
df = pd.read_csv('/content/dataset/training_data_processed.csv')
drop_cols = ['evasion_label', 'Unnamed: 0', 'affirmative_questions']
df = df.drop(columns= drop_cols)
df.head()

,question_order,interview_question,interview_answer,gpt3.5_summary,gpt3.5_prediction,question,clarity_label
0,1,Q. Of the Biden administration. And accused th...,"Well, look, first of all, theI am sincere abou...",The question consists of 2 parts: \n1. How wou...,Question part: 1. How would you respond to the...,How would you respond to the accusation that t...,Clear Reply
1,1,Q. Of the Biden administration. And accused th...,"Well, look, first of all, theI am sincere abou...",The question consists of 2 parts: \n1. How wou...,Question part: 1. How would you respond to the...,Do you think President Xi is being sincere abo...,Ambivalent
2,2,Q. No worries. Do you believe the country's sl...,"Look, I think China has a difficult economic p...",The question consists of two parts:\n\n1. Q1: ...,Question part: Q1 - Do you believe the country...,Do you believe the country's slowdown and gro...,Ambivalent
3,2,Q. No worries. Do you believe the country's sl...,"Look, I think China has a difficult economic p...",The question consists of two parts:\n\n1. Q1: ...,Question part: Q1 - Do you believe the country...,Are you worried about the meeting between Pre...,Ambivalent
4,3,"Q. I can imagine. It is evening, I'd like to r...","Well, I hope I get to see Mr. Xi sooner than l...",The question consists of 3 parts:\n1. Is the P...,Question part: 1. Is the President's engagemen...,Is the President's engagement with Asian coun...,Clear Reply


In [3]:
label_map = {
    'Clear Reply': 0,
    'Clear Non-Reply': 1,
    'Ambivalent': 2
}
num_labels = len(label_map)
model_name = 'bert-base-uncased'


modelBert = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels) # For binary classification
tokenizerBert = BertTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
def data_prep(questions, answers, clarity_label):
    encodings = tokenizerBert(questions.tolist(),
                          answers.tolist(),
                          padding= True,
                          truncation = True,
                          max_length = 512,
                          return_tensors = 'pt'
                          )
    label_tensors = torch.tensor([label_map[l] for l in clarity_label.tolist()])
    dataset = TensorDataset(
        encodings['input_ids'],
        encodings['attention_mask'],
        encodings['token_type_ids'],
        label_tensors
        )
    return dataset


In [5]:
from huggingface_hub import login

login()

In [6]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

# Configuration for 4-bit quantization (memory efficiency)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto", # Automatically place the model on the T4 GPU
    token=True         # Uses the token you logged in with
)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [10]:
SYSTEM_PROMPT = """
You are a highly accurate text extraction assistant. Your task is to analyze a long interview answer and a specific subquestion.

You must identify the exact sentence or sentences within the provided interview answer that directly address the subquestion.

If the subquestion is not answered, return only the phrase "NOT_ANSWERED".

Do not paraphrase, summarize, or add any extra commentary. Return only the extracted text or the "NOT_ANSWERED" phrase.
"""

SYSTEM_PROMPT_SUMMARY = """
You are a highly accurate text extraction assistant. Your task is to analyze a long interview answer and a specific subquestion.

You must identify the part in the answer in which the question is adressed and summarize it.

For example:
  Question: How would you respond to the accusation that the United States is containing China while pushing for diplomatic talks?

  Interview answer: Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things.And I think that one of the things we've doneI've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stability.That's what this trip was all about: having India cooperate much more with the United States, be closer with the United States, Vietnam being closer with the United States. It's not about containing China; it's about having a stable base, a stable base in the Indo-Pacific.And it'sfor example, when I was spending a lot of time talking with President Xi, he asked why we were doingwhy was I going to have the Quad, meaning Australia, India, Japan, and the United States? And I said, To maintain stability. It's not about isolating China. It's about making sure the rules of the roadeverything from airspace and space in the ocean isthe international rules of the road are abided by.And soand I hope thatI think that Prime Minister XiI mean, Xi has somesome difficulties right now. All countries end up with difficulties, and he had some economic difficulties he's working his way through. I want to see China succeed economically, but I want to see them succeed by the rules.The next question was to Bloomberg.

  Summarized answer: The President expresses sincerity about getting the relationship between the United States and China right.


Return only the summary.
"""

def get_answers_llama3(question, full_answer):
    user_prompt = f"""
Subquestion: "{question}"

Full Interview Answer:
---
{full_answer}
---

Extracted Part:
"""

    # Llama 3 uses a specific chat format which is applied here
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    # Apply the template and move tensors to the GPU
    model_inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_tensors = model.generate(
            model_inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id # Prevents generation issues with padding
        )

    # Decode the response, skipping the input tokens we already passed in
    output_text = tokenizer.decode(output_tensors[0][model_inputs.shape[1]:], skip_special_tokens=True).strip()
    return output_text

# Example test before running on the full dataframe
test_q = "What is the policy?"
test_a = "Our policy is clearly defined in section 3. We updated it last year after feedback."
print(f"Extracted: {get_answers_llama3(test_q, test_a)}")

Extracted: "Our policy is clearly defined in section 3. We updated it last year after feedback."


In [13]:
import os
from tqdm.auto import tqdm

processed_df = df.copy()
processed_df['answer'] = None # Initialize a new column

# Define where to save checkpoints
CHECKPOINT_FILE = '/content/dataset/ckpt_extracted_answers.csv'
CHECKPOINT_INTERVAL = 200
DISPLAY_INTERVAL = 100

if os.path.exists(CHECKPOINT_FILE):
    print(f"Checkpoint found. Resuming from {CHECKPOINT_FILE}")
    processed_df = pd.read_csv(CHECKPOINT_FILE)
    last_processed_index = processed_df['answer'].last_valid_index()
    if last_processed_index is not None:
        start_index = last_processed_index + 1
    else:
        start_index = 0
else:
    start_index = 0


print(f"Starting extraction from entry {start_index} of {len(processed_df)}")

for index, row in tqdm(processed_df.iloc[start_index:].iterrows(), total=len(processed_df.iloc[start_index:])):

    question = row['question']
    interview_answer = row['interview_answer']

    # 3. Call your extraction function
    extracted_text = get_answers_llama3(question, interview_answer)

    # 4. Save the result into the temporary DataFrame copy
    processed_df.at[index, 'answer'] = extracted_text

    if (index + 1) % DISPLAY_INTERVAL == 0:
        print(f"\n--- Sample Display at Entry {index + 1} ---")
        print(f"Question: {question[:100]}...")
        print(f"Full Answer (Snippet): {interview_answer[:150]}...")
        print(f"**Extracted Answer:** {extracted_text}")
        print("------------------------------------------")


    # 5. Checkpoint Saving Logic
    if (index + 1) % CHECKPOINT_INTERVAL == 0:
        processed_df.to_csv(CHECKPOINT_FILE, index=False)
        print(f"\nCheckpoint saved at entry {index + 1}.")

print("Processing complete!")

FINAL_FILE_NAME = 'training_data_preprocessed_LLM.csv'
processed_df.to_csv(FINAL_FILE_NAME, index=False)
print(f"Final data saved to {FINAL_FILE_NAME}")








Starting extraction from entry 0 of 3448


  0%|          | 0/3448 [00:00<?, ?it/s]


--- Sample Display at Entry 100 ---
Question: Evaluating whether President Xi was more confrontational or conciliatory and willing to compromise d...
Full Answer (Snippet): Neither. And yes....
**Extracted Answer:** "Neither."
------------------------------------------

--- Sample Display at Entry 200 ---
Question: Is deescalation still possible given the presence of 100,000 troops at the border?...
Full Answer (Snippet): The answer is yes....
**Extracted Answer:** NOT_ANSWERED
------------------------------------------

Checkpoint saved at entry 200.

--- Sample Display at Entry 300 ---
Question: Reasons for the shortened duration of the summit...
Full Answer (Snippet): Yes....
**Extracted Answer:** NOT_ANSWERED
------------------------------------------

--- Sample Display at Entry 400 ---
Question: To follow on that, if he tries to dodge the questions or doesn't address them head on, are you prepa...
Full Answer (Snippet): Well, I have all the information. It just came out. I mean,

In [17]:

from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/training_data_preprocessed_LLM.csv')
# Assuming 'df' is your full DataFrame
Q_train, Q_val, A_train, A_val, L_train, L_val = train_test_split(
    df['question'],
    df['answer'],
    df['clarity_label'],
    test_size=0.2,
    random_state=40
    )



BATCH_SIZE = 16

train_dataset = data_prep(Q_train, A_train, L_train)
val_dataset = data_prep(Q_val, A_val, L_val)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

### Model Training

With preprocessing done we can now gop on to build the training loop to finetune the BERT model

In [20]:
import numpy as np
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, classification_report



model = modelBert

def eval_model(model, data_loader, device):
    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids, attention_mask, token_type_ids, labels = [t.to(device) for t in batch]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )

            _, predicted = torch.max(outputs.logits, 1)

            # Store labels and predictions (move from GPU to CPU/Numpy)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    model.train() # Set back to training mode

    accuracy = 100 * np.sum(np.array(all_predictions) == np.array(all_labels)) / len(all_labels)
    f1_weighted = f1_score(all_labels, all_predictions, average='weighted')
    conf_matrix = confusion_matrix(all_labels, all_predictions)

    class_report = classification_report(all_labels, all_predictions, digits=4, output_dict=True)

    return accuracy, f1_weighted, conf_matrix, class_report


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
epochs = 20
optimizer = AdamW(model.parameters(), lr = 5e-5)



for epoch in range(epochs):
    print(f"--- Starting Epoch {epoch+1}/{epochs} ---")
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        token_type_ids = batch[2].to(device)
        labels = batch[3].to(device)

        model.zero_grad()

        outputs = model(input_ids= input_ids,
                        attention_mask = attention_mask,
                        token_type_ids = token_type_ids,
                        labels = labels
                        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    val_accuracy, val_f1, val_conf_matrix, val_report = eval_model(model, val_loader, device)

    print(f"Epoch {epoch+1} Summary:")
    print(f"  Training Loss: {avg_train_loss:.4f}")
    print(f"  Validation Accuracy: {val_accuracy:.2f}%")
    print(f"  Validation F1 (Weighted): {val_f1:.4f}")

print("\n  --- Confusion Matrix (Validation) ---")
# A DataFrame makes the confusion matrix much easier to read
# Assuming your labels map is defined as {'clear_reply': 0, 'clear_non_reply': 1, 'ambivalent_reply': 2}
labels_list = list(label_map.keys())
cm_df = pd.DataFrame(val_conf_matrix, index=labels_list, columns=labels_list)
print(cm_df)

--- Starting Epoch 1/20 ---
Epoch 1 Summary:
  Training Loss: 0.0352
  Validation Accuracy: 52.17%
  Validation F1 (Weighted): 0.5218
--- Starting Epoch 2/20 ---
Epoch 2 Summary:
  Training Loss: 0.0190
  Validation Accuracy: 57.97%
  Validation F1 (Weighted): 0.5723
--- Starting Epoch 3/20 ---
Epoch 3 Summary:
  Training Loss: 0.0242
  Validation Accuracy: 54.49%
  Validation F1 (Weighted): 0.5464
--- Starting Epoch 4/20 ---
Epoch 4 Summary:
  Training Loss: 0.0284
  Validation Accuracy: 56.38%
  Validation F1 (Weighted): 0.5518
--- Starting Epoch 5/20 ---
Epoch 5 Summary:
  Training Loss: 0.0178
  Validation Accuracy: 59.71%
  Validation F1 (Weighted): 0.5779
--- Starting Epoch 6/20 ---
